# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et de combiner **un dataset multi-tables ou plusieurs datasets publics complémentaires sur Kaggle** afin de les croiser par jointure.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner **un ou plusieurs datasets Kaggle** (~50 à 500 Mo au total) partageant au moins un axe de jointure (date/heure, zone géographique, code ISO/catégorie, identifiant métier).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts multi-sources sur HDFS** (`/lakehouse/bronze/raw/<source>/...`) et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** avec traçabilité de la source (`_source_dataset`, `_source_file`, `_ingested_at`) et inspecter vos jobs sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Titre du sujet :** `Prédire la probabilité de recevoir un commentaire écrit après la livraison.`
- **Dataset(s) Kaggle retenu(s) (`auteur/nom-du-dataset`) :*olistbr/brazilian-ecommerce*
  1. Source principale : `olist_orders_dataset.csv`
  2. Source(s) complémentaire(s) pour jointure : `olist_order_items_dataset.csv` pour identifier l’article acheté, `olist_products_dataset.csv` pour sa catégorie et ses caractéristiques,`olist_order_reviews_dataset.csv` pour repérer les commentaires écrits.
- **Clé(s) de jointure envisagée(s) entre sources / tables :** `order_id` *(ex. date + ville, code aéroport, code postal, catégorie...)*
- **Problématique métier principale :** `Au moment de la livraison, peut-on prédire la probabilité qu’un client laisse un commentaire écrit à partir de la catégorie du produit, de son prix et des conditions de livraison ?`
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :** 
- Temporelles — Date de commande (order_purchase_timestamp), date réelle de livraison (order_delivered_customer_date) et date promise (order_estimated_delivery_date).
- Quantitatives — Prix de l’article (price), frais de livraison (freight_value), poids du produit (product_weight_g), délai réel de livraison et écart en jours par rapport à la date promise.
- Catégorielles — Catégorie du produit (product_category_name) et statut de livraison « à temps » ou « en retard ».
- Variable cible — a_un_commentaire vaut 1 si au moins un avis de la commande contient un message écrit (review_comment_message), et 0 sinon.

## 1. Préparer les CSV Olist dans `data_raw/`

Le projet utilise le jeu de données Kaggle `olistbr/brazilian-ecommerce`. La cellule cherche les CSV à la racine de `data_raw/`, puis dans `data_raw/brazilian-ecommerce/`. Si aucun de ces emplacements ne contient les quatre tables nécessaires, elle télécharge le dataset dans le sous-dossier.

Les quatre tables du projet sont les commandes, les articles commandés, les produits et les avis. Le fichier de géolocalisation, s’il est présent, sert aussi à observer le découpage en blocs HDFS de 16 Mio.

In [20]:
from pathlib import Path
from shutil import copy2

current_dir = Path.cwd()
project_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents, Path("/workspace"))
        if (folder / "docker-compose.yml").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Ouvre le dossier hadoop-kaggle dans VS Code ou exécute ce notebook dans JupyterLab."
    )

DATA_RAW = project_root / "data_raw"
KAGGLE_DATASET = "olistbr/brazilian-ecommerce"
REQUIRED_CSVS = {
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_order_reviews_dataset.csv",
}


def missing_csvs(folder):
    return sorted(name for name in REQUIRED_CSVS if not (folder / name).is_file())


kaggle_dir = DATA_RAW / "brazilian-ecommerce"
source_dir = DATA_RAW if not missing_csvs(DATA_RAW) else kaggle_dir

if missing_csvs(source_dir):
    import kagglehub

    print(f"Téléchargement de {KAGGLE_DATASET} vers {kaggle_dir} ...")
    cache_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    missing = missing_csvs(cache_dir)
    if missing:
        raise FileNotFoundError(f"CSV Olist manquants : {', '.join(missing)}")

    kaggle_dir.mkdir(parents=True, exist_ok=True)
    for csv_file in cache_dir.glob("*.csv"):
        target = kaggle_dir / csv_file.name
        if not target.exists():
            copy2(csv_file, target)

local_files = sorted(
    file for file in source_dir.glob("*.csv")
    if file.is_file() and (
        file.name.startswith("olist_")
        or file.name == "product_category_name_translation.csv"
    )
)

block_size = 16 * 1024 * 1024
print(f"Inventaire Olist dans {source_dir} ({len(local_files)} CSV) :")
for file in local_files:
    size = file.stat().st_size
    expected_blocks = (size + block_size - 1) // block_size
    print(f" - {str(file.relative_to(DATA_RAW)):<55} {size / 1024**2:>7.2f} Mio  (~{expected_blocks} bloc(s) HDFS)")

Inventaire Olist dans c:\Users\antoi\Desktop\cour\MASTER 2\SEANCE 1\hadoop-kaggle\data_raw (9 CSV) :
 - olist_customers_dataset.csv                                8.62 Mio  (~1 bloc(s) HDFS)
 - olist_geolocation_dataset.csv                             58.44 Mio  (~4 bloc(s) HDFS)
 - olist_order_items_dataset.csv                             14.72 Mio  (~1 bloc(s) HDFS)
 - olist_order_payments_dataset.csv                           5.51 Mio  (~1 bloc(s) HDFS)
 - olist_order_reviews_dataset.csv                           13.78 Mio  (~1 bloc(s) HDFS)
 - olist_orders_dataset.csv                                  16.84 Mio  (~2 bloc(s) HDFS)
 - olist_products_dataset.csv                                 2.27 Mio  (~1 bloc(s) HDFS)
 - olist_sellers_dataset.csv                                  0.17 Mio  (~1 bloc(s) HDFS)
 - product_category_name_translation.csv                      0.00 Mio  (~1 bloc(s) HDFS)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

**Dans VS Code**, exécutez le point 1, puis la cellule de connexion ci-dessous. Les cellules `%%spark_hdfs` utilisent ensuite le conteneur `spark-jupyter` avec la liste de fichiers préparée au point 1. Docker et le cluster doivent être démarrés.

In [21]:
import atexit
import json
import subprocess
from pathlib import Path
from uuid import uuid4

shell = get_ipython()
in_spark_container = Path("/workspace/docker-compose.yml").is_file()
docker_process = globals().get("docker_process")

# Le processus Python du conteneur conserve les variables entre les cellules.
docker_worker = """
import json
import sys
import traceback

namespace = {"__name__": "__main__"}
for line in sys.stdin:
    request = json.loads(line)
    try:
        exec(compile(request["code"], "<cellule Spark/HDFS>", "exec"), namespace)
    except Exception:
        traceback.print_exc()
        success = False
    else:
        success = True
    print(f'\\n{request["marker"]}{int(success)}', flush=True)
"""


def run_spark_cell(line, cell):
    global docker_process
    if in_spark_container:
        shell.run_cell(cell, store_history=False).raise_error()
        return

    if docker_process is None:
        try:
            docker_process = subprocess.Popen(
                ["docker", "exec", "-i", "-w", "/workspace",
                 "spark-jupyter", "python3", "-u", "-c", docker_worker],
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding="utf-8",
                errors="replace",
                creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
            )
        except FileNotFoundError as exc:
            raise RuntimeError("Docker doit être disponible pour exécuter ce notebook.") from exc
        atexit.register(docker_process.stdin.close)

    marker = "spark-cell-" + uuid4().hex + ":"
    try:
        docker_process.stdin.write(json.dumps({"code": cell, "marker": marker}) + "\n")
        docker_process.stdin.flush()
    except OSError as exc:
        raise RuntimeError("La session Spark Docker est arrêtée. Redémarre le kernel et reprends depuis la première cellule.") from exc

    for output in docker_process.stdout:
        if output.startswith(marker):
            if output.strip() == marker + "1":
                return
            raise RuntimeError("La cellule a échoué dans Spark. Consulte le détail affiché ci-dessus.")
        print(output, end="")
    raise RuntimeError("Le conteneur spark-jupyter ne répond plus. Vérifie Docker et redémarre le kernel.")


shell.register_magic_function(run_spark_cell, magic_kind="cell", magic_name="spark_hdfs")
if not in_spark_container:
    if "local_files" not in globals():
        raise RuntimeError("Exécute d’abord le point 1 pour préparer la liste des CSV.")
    relative_files = [file.relative_to(DATA_RAW).as_posix() for file in local_files]
    run_spark_cell(
        "",
        "from pathlib import Path\n"
        "DATA_RAW = Path('/workspace/data_raw')\n"
        f"local_files = [DATA_RAW / name for name in {relative_files!r}]",
    )

print("Cellules Spark/HDFS prêtes.")

  File "<string>", line 10, in <module>
    exec(compile(request["code"], "<cellule Spark/HDFS>", "exec"), namespace)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<cellule Spark/HDFS>", line 8, in <module>
AssertionError: Aucune table Delta détectée dans /lakehouse/bronze/ !

Cellules Spark/HDFS prêtes.


In [22]:
%%spark_hdfs
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("J1_HDFS_Ingestion_Bronze").getOrCreate()

default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = int(spark.conf.get("spark.hadoop.dfs.blocksize"))
replication = int(spark.conf.get("spark.hadoop.dfs.replication"))

print("Spark version         :", spark.version)
print("Système de fichiers   :", default_fs)
print(f"Taille d'un bloc HDFS : {block_size:,} octets ({block_size / 1024**2:g} Mio)")
print("Réplication HDFS       :", replication)

Spark version         : 4.2.0
Système de fichiers   : hdfs://namenode:9000
Taille d'un bloc HDFS : 16,777,216 octets (16 Mio)
Réplication HDFS       : 2



## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [23]:
%%spark_hdfs
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(file_path.as_uri())
    dst = PathHDFS(raw_hdfs_dir, rel_posix)
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/olist_customers_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist_geolocation_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist_order_items_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist_order_payments_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist_order_reviews_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist_orders_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist_products_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/olist_sellers_dataset.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/product_category_name_translation.csv



In [24]:
%%spark_hdfs
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
if not statuses:
    raise FileNotFoundError(f"Aucun fichier trouvé dans {raw_hdfs_dir} sur HDFS.")
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : olist_geolocation_dataset.csv (58.44 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #3 (10.44 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : olist_orders_dataset.csv (16.84 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #1 ( 0.84 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : olist_order_items_dataset.csv (14.72 Mo | repl=2)
   Bloc #0 (14.72 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : olist_order_reviews_dataset.csv (13.78 Mo | repl=2)
   Bloc #0 (13.78 Mo) -> DataNodes : ['datanode2', 'datanode1']
Fichier : olist_customers_dataset.csv (8.62 Mo | repl=2)
   Bloc #0 ( 8.62 Mo) -> DataNodes : ['datanode3', 'datanode2']
Fichier : olist_order_payments_dataset.csv (5.51 Mo | repl=2)
   Bloc #0 ( 5.51 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fich

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

**Réponses**

1. Le fichier `olist_geolocation_dataset.csv` occupe environ **58,44 Mio**. HDFS le répartit sur **4 blocs** : trois blocs de 16 Mio et un dernier d’environ 10,44 Mio.
2. Un fichier d’environ 3 Mio utilise environ 3 Mio par copie. Le dernier bloc stocke uniquement les octets du fichier. Avec un facteur de réplication de 2, cela représente environ 6 Mio de données au total.
3. HDFS place deux copies de chaque bloc sur deux DataNodes distincts. Si l’un tombe en panne, l’autre copie reste lisible et le troisième DataNode peut héberger la copie de remplacement.

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

**Interprétation.** Les blocs sont les unités de stockage et de réplication HDFS. Spark définit ses partitions de lecture selon la taille du fichier et sa configuration : il peut lire un bloc avec plusieurs tâches ou regrouper des portions de fichier. La lecture reste possible tant qu’une copie de chaque bloc est accessible.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [25]:
%%spark_hdfs
print("Fichier cible du test de résilience :", largest_hdfs_path)

file_status = fs.getFileStatus(PathHDFS(largest_hdfs_path))
block_locations = fs.getFileBlockLocations(file_status, 0, file_status.getLen())
df_check = spark.read.option("header", True).csv(largest_hdfs_path)

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de blocs HDFS              : {len(block_locations)}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/olist_geolocation_dataset.csv
Nombre total de lignes lues       : 1,000,163
Nombre de blocs HDFS              : 4
Nombre de partitions Spark créées : 12



---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

La cellule crée les tables Delta `olist_orders`, `olist_order_items`, `olist_products` et `olist_order_reviews`. Les avis sont lus en mode multiligne pour préserver les retours à la ligne dans les commentaires.

En relançant la cellule, les quatre tables sont reconstruites depuis les CSV bruts.

In [26]:
%%spark_hdfs
BRONZE_SOURCES = {
    "olist_orders": "olist_orders_dataset.csv",
    "olist_order_items": "olist_order_items_dataset.csv",
    "olist_products": "olist_products_dataset.csv",
    "olist_order_reviews": "olist_order_reviews_dataset.csv",
}

source_paths = {file.name: file.relative_to(DATA_RAW).as_posix() for file in local_files}
missing = sorted(set(BRONZE_SOURCES.values()) - source_paths.keys())
if missing:
    raise FileNotFoundError(f"CSV Olist manquants : {', '.join(missing)}")

for table_name, filename in BRONZE_SOURCES.items():
    raw_path = PathHDFS(raw_hdfs_dir, source_paths[filename]).toString()
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"
    df_raw = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .option("escape", '"')
        .option("multiLine", table_name == "olist_order_reviews")
        .csv(raw_path)
    )
    df_bronze = (
        df_raw.withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_source_dataset", F.lit("olistbr/brazilian-ecommerce"))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    (
        df_bronze.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .save(bronze_delta_path)
    )
    count = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path} → {count:,} lignes")

26/11/16 16:33:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.

[Stage 8:>                                                        (0 + 12) / 50]

[Stage 8:=============>                                          (12 + 12) / 50]

[Stage 8:=================>                                      (16 + 12) / 50]

[Stage 8:==========================>                             (24 + 12) / 50]

[Stage 8:================================>                       (29 + 12) / 50]

[Stage 8:======================================>                 (34 + 12) / 50]

[Stage 8:============================================>           (40 + 10) / 50]

                                                                                

[Stage 12:>                                                         (0 + 5) / 5]

[Stage 12:===========>                                              (1 +

In [27]:
%%spark_hdfs
# Auto-contrôle : vérifier que les quatre tables Olist existent en Delta.
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    status.getPath().getName()
    for status in fs.listStatus(bronze_root)
    if status.isDirectory() and fs.exists(PathHDFS(status.getPath(), "_delta_log"))
]
missing_tables = sorted(set(BRONZE_SOURCES) - set(bronze_delta_tables))
assert not missing_tables, f"Tables Delta Olist manquantes : {', '.join(missing_tables)}"
print("Tables Delta Bronze validées :", ", ".join(sorted(BRONZE_SOURCES)))

Tables Delta Bronze validées : olist_order_items, olist_order_reviews, olist_orders, olist_products

